# Lab 14 - Multi-label classification


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

from itertools import combinations
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, hamming_loss, jaccard_score
from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.multioutput import ClassifierChain
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


In [ ]:
X = np.genfromtxt("../../tasks/datasets/emotions_X.csv", delimiter=",", skip_header=1)
Y = np.genfromtxt("../../tasks/datasets/emotions_Y.csv", delimiter=",", skip_header=1).astype(int)
label_names = open("../../tasks/datasets/emotions_Y.csv", encoding="utf-8").readline().strip().split(",")
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, random_state=RANDOM_STATE)
print(X.shape, Y.shape, label_names)


## Task 1 - preliminary label analysis


In [ ]:
print("number of labels:", Y.shape[1])
print("label frequencies:")
for name, freq in zip(label_names, Y.mean(axis=0)):
    print(f"{name:25s} {freq:.3f}")
print("average active labels per observation:", Y.sum(axis=1).mean())

corr = np.corrcoef(Y, rowvar=False)
pairs = []
for i, j in combinations(range(Y.shape[1]), 2):
    pairs.append((abs(corr[i, j]), corr[i, j], label_names[i], label_names[j]))
for _, value, a, b in sorted(pairs, reverse=True)[:5]:
    print(f"{a} / {b}: corr={value:.3f}")


The labels are not independent when correlations are visibly nonzero. This matters because binary relevance ignores label dependence, while chains and CCC attempt to exploit it.


## Task 2 - BR, classifier chains, and ensemble of classifier chains


In [ ]:
def base_lr():
    return make_pipeline(StandardScaler(), LogisticRegression(solver="liblinear", max_iter=1000))


def evaluate(name, Y_true, Y_pred):
    return {
        "method": name,
        "subset_accuracy": accuracy_score(Y_true, Y_pred),
        "hamming_score": 1 - hamming_loss(Y_true, Y_pred),
        "jaccard_score": jaccard_score(Y_true, Y_pred, average="samples", zero_division=0),
    }


br = OneVsRestClassifier(base_lr()).fit(X_train, Y_train)
Y_br = br.predict(X_test)
rows = [evaluate("BR", Y_test, Y_br)]

chain_preds = []
for seed in range(10):
    chain = ClassifierChain(base_lr(), order="random", random_state=seed).fit(X_train, Y_train)
    pred = chain.predict(X_test).astype(int)
    chain_preds.append(pred)
    rows.append(evaluate(f"CC_{seed}", Y_test, pred))

ecc_preds = []
for seed in range(20):
    chain = ClassifierChain(base_lr(), order="random", random_state=100 + seed).fit(X_train, Y_train)
    ecc_preds.append(chain.predict(X_test).astype(int))
Y_ecc = (np.mean(ecc_preds, axis=0) >= 0.5).astype(int)
rows.append(evaluate("ECC", Y_test, Y_ecc))
rows


## Task 3 - Circular Chain Classifier


In [ ]:
def fit_ccc(X, Y):
    models = []
    K = Y.shape[1]
    for k in range(K):
        Y_other = np.delete(Y, k, axis=1)
        X_aug = np.column_stack([X, Y_other])
        model = base_lr().fit(X_aug, Y[:, k])
        models.append(model)
    return models


def predict_ccc(models, X, init="zeros", br_model=None, order=None, max_iter=20, random_state=42):
    local_rng = np.random.default_rng(random_state)
    n = X.shape[0]
    K = len(models)
    if init == "zeros":
        Y_hat = np.zeros((n, K), dtype=int)
    elif init == "random":
        Y_hat = local_rng.integers(0, 2, size=(n, K))
    elif init == "br":
        if br_model is None:
            raise ValueError("br_model is required for BR initialization")
        Y_hat = br_model.predict(X).astype(int)
    else:
        raise ValueError("unknown init")

    order = np.arange(K) if order is None else np.asarray(order)
    converged = np.zeros(n, dtype=bool)
    iterations = np.zeros(n, dtype=int)

    for it in range(1, max_iter + 1):
        old = Y_hat.copy()
        for k in order:
            Y_other = np.delete(Y_hat, k, axis=1)
            X_aug = np.column_stack([X, Y_other])
            Y_hat[:, k] = (models[k].predict_proba(X_aug)[:, 1] >= 0.5).astype(int)
        changed = np.any(Y_hat != old, axis=1)
        newly_done = (~changed) & (~converged)
        iterations[newly_done] = it
        converged |= ~changed
        if not changed.any():
            break
    iterations[iterations == 0] = max_iter
    return Y_hat, converged, iterations


In [ ]:
ccc_models = fit_ccc(X_train, Y_train)
ccc_rows = []
for init in ["zeros", "random", "br"]:
    pred, conv, iters = predict_ccc(ccc_models, X_test, init=init, br_model=br, random_state=RANDOM_STATE)
    ccc_rows.append({**evaluate(f"CCC_{init}", Y_test, pred), "convergence_rate": conv.mean(), "avg_iterations": iters.mean()})

for seed in range(5):
    order = np.random.default_rng(seed).permutation(Y.shape[1])
    pred, conv, iters = predict_ccc(ccc_models, X_test, init="br", br_model=br, order=order, random_state=seed)
    ccc_rows.append({**evaluate(f"CCC_order_{seed}", Y_test, pred), "convergence_rate": conv.mean(), "avg_iterations": iters.mean()})

rows + ccc_rows


Questions:

1. Binary relevance assumes conditional label independence given X.
2. Classifier chains depend on order because early prediction errors become later input variables.
3. ECC is more stable because it averages over many random orders.
4. Subset accuracy is smaller because one wrong label makes the whole vector wrong.
5. Jaccard focuses on active labels, so it is more informative when positives are rare.
6. CCC searches for a self-consistent vector because each label is repeatedly updated using the current values of the other labels.
7. CCC defines a collection of conditional models, not necessarily one coherent joint distribution. That matters because incompatible conditionals may not correspond to any valid joint model.
